# Analysis 260304: Data Validation & Enhanced Research Analysis

**Dataset**: `coldstart_dataset_260304.csv` (438,519 users × 154 cols)

## Phases
1. Data Load & Validation (new dataset vs old)
2. Full Re-verification of research_framing_kor.md numbers
3. 30-minute Resolution Time Analysis (new M60-M360 columns)
4. LTV Analysis
5. Creative Analysis Integration
6. QA Cross-checks

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus'] = False

print('Libraries loaded.')

---
## Phase 1: Data Load & Validation

In [ ]:
# Load new dataset (260304) - LTV column contains multi-line JSON
df_new = pd.read_csv('coldstart_dataset_260304.csv', low_memory=False)
print(f'New dataset: {df_new.shape[0]:,} rows × {df_new.shape[1]} cols')
print(f'Expected: 438,519 rows × 154 cols')
assert df_new.shape[0] == 438519, f'Row count mismatch: {df_new.shape[0]}'
assert df_new.shape[1] == 154, f'Col count mismatch: {df_new.shape[1]}'
print('✓ Shape validated')

In [ ]:
# Load old dataset (260301) for comparison
df_old = pd.read_csv('coldstart_dataset_260301.csv', low_memory=False)
print(f'Old dataset: {df_old.shape[0]:,} rows × {df_old.shape[1]} cols')

# Identify column differences
old_cols = set(df_old.columns)
new_cols = set(df_new.columns)

added = sorted(new_cols - old_cols)
removed = sorted(old_cols - new_cols)
shared = sorted(old_cols & new_cols)

print(f'\nShared columns: {len(shared)}')
print(f'Added columns ({len(added)}): {added}')
print(f'Removed columns ({len(removed)}): {removed}')

In [ ]:
# Validate H1↔M60, H6↔M360, H24↔D1 mapping
# Merge on USER_ID to compare
df_compare = df_old[['USER_ID', 'IS_H1_PURCHASE', 'IS_H6_PURCHASE', 'IS_H24_PURCHASE',
                      'IS_H1_CHURN', 'IS_H6_CHURN', 'IS_H24_CHURN']].merge(
    df_new[['USER_ID', 'IS_M60_PURCHASE', 'IS_M360_PURCHASE', 'IS_D1_PURCHASE',
            'IS_M60_CHURN', 'IS_M360_CHURN', 'IS_D1_CHURN']], on='USER_ID', how='inner')

print(f'Matched users: {len(df_compare):,}')

mappings = [
    ('IS_H1_PURCHASE', 'IS_M60_PURCHASE', 'H1↔M60 Purchase'),
    ('IS_H6_PURCHASE', 'IS_M360_PURCHASE', 'H6↔M360 Purchase'),
    ('IS_H24_PURCHASE', 'IS_D1_PURCHASE', 'H24↔D1 Purchase'),
    ('IS_H1_CHURN', 'IS_M60_CHURN', 'H1↔M60 Churn'),
    ('IS_H6_CHURN', 'IS_M360_CHURN', 'H6↔M360 Churn'),
    ('IS_H24_CHURN', 'IS_D1_CHURN', 'H24↔D1 Churn'),
]

for old_col, new_col, label in mappings:
    match = (df_compare[old_col] == df_compare[new_col]).mean()
    diff_count = (df_compare[old_col] != df_compare[new_col]).sum()
    print(f'{label}: {match*100:.2f}% match ({diff_count:,} differences)')

In [ ]:
# Validate fraud count and analysis population
fraud_count = df_new['IS_HAS_FRAUD'].sum()
analysis_pop = len(df_new) - fraud_count
print(f'Total users: {len(df_new):,}')
print(f'Fraud users: {fraud_count:,} (expected: 53,494)')
print(f'Analysis population: {analysis_pop:,} (expected: 385,025)')

# Create analysis subset
df = df_new[df_new['IS_HAS_FRAUD'] == 0].copy()
print(f'\nAnalysis df: {len(df):,} rows')

In [ ]:
# Verify shared column distributions match
# Compare key target columns between old and new datasets
df_old_clean = df_old[df_old['IS_HAS_FRAUD'] == 0].copy()

check_cols = ['IS_D7_PURCHASE', 'IS_D30_PURCHASE', 'IS_M10_PURCHASE', 'IS_M30_PURCHASE',
              'IS_D7_CHURN', 'IS_D30_CHURN', 'IS_M10_CHURN', 'IS_M30_CHURN']

print('Column distribution comparison (old vs new, fraud-excluded):')
print(f'{"Column":<20} {"Old mean":>10} {"New mean":>10} {"Match":>8}')
print('-' * 52)
for col in check_cols:
    if col in df_old_clean.columns and col in df.columns:
        old_mean = df_old_clean[col].mean()
        new_mean = df[col].mean()
        match = '✓' if abs(old_mean - new_mean) < 0.001 else '✗'
        print(f'{col:<20} {old_mean:>10.4f} {new_mean:>10.4f} {match:>8}')

In [ ]:
# Parse LTV JSON
print('LTV column sample values:')
print(df['ltv'].head(10).tolist())
print(f'\nLTV non-null count: {df["ltv"].notna().sum():,}')
print(f'LTV null count: {df["ltv"].isna().sum():,}')

# Try to parse LTV
def parse_ltv(x):
    if pd.isna(x):
        return {}
    try:
        if isinstance(x, str):
            return json.loads(x)
        return {}
    except:
        return {}

ltv_parsed = df['ltv'].apply(parse_ltv)
ltv_keys = set()
for d in ltv_parsed.head(1000):
    ltv_keys.update(d.keys())
print(f'\nLTV JSON keys found: {sorted(ltv_keys)}')

# Show a few parsed examples
for i, val in enumerate(ltv_parsed[:20]):
    if val:
        print(f'  User {i}: {val}')
        break

---
## Phase 2: Full Re-verification of research_framing_kor.md Numbers

### Section 1: Golden Time (Churn Rates)

In [ ]:
# Parse inapp JSON columns
# FIX: Exclude purchase_engagement (data leakage) and compute adjusted_totalEventCount
INAPP_WINDOWS = ['m10', 'm30', 'm60', 'm90', 'm120', 'm150', 'm180',
                 'm210', 'm240', 'm270', 'm300', 'm330', 'm360',
                 'd1', 'd2', 'd3', 'd7', 'd14', 'd30']

# Raw keys for parsing (includes purchase_engagement for subtraction only)
INAPP_KEYS_RAW = ['active', 'ad_engagement', 'core_engagement', 'deeplink_count',
                  'open_count', 'purchase_engagement', 'totalEventCount']

# Clean keys for modeling (no purchase_engagement, adjusted totalEventCount)
INAPP_KEYS = ['active', 'ad_engagement', 'core_engagement', 'deeplink_count',
              'open_count', 'adjusted_totalEventCount']

for window in INAPP_WINDOWS:
    col = f'inapp_{window}'
    if col in df.columns:
        parsed = df[col].apply(lambda x: json.loads(x) if pd.notna(x) and isinstance(x, str) else {})
        for key in INAPP_KEYS_RAW:
            df[f'inapp_{window}_{key}'] = parsed.apply(lambda d: d.get(key, 0))
        # Compute adjusted_totalEventCount = totalEventCount - purchase_engagement
        df[f'inapp_{window}_adjusted_totalEventCount'] = (
            df[f'inapp_{window}_totalEventCount'] - df[f'inapp_{window}_purchase_engagement']
        )
        # Drop raw columns not used in modeling
        df.drop(columns=[col, f'inapp_{window}_purchase_engagement', f'inapp_{window}_totalEventCount'], inplace=True)

def get_inapp_features(window):
    """Return clean InApp feature column names for a given time window."""
    return [f'inapp_{window}_{k}' for k in INAPP_KEYS]

print(f'InApp features parsed (leakage-safe). Total columns now: {len(df.columns)}')
print(f'InApp windows: {len(INAPP_WINDOWS)}')
print(f'Clean InApp keys per window: {len(INAPP_KEYS)}')
print(f'Total InApp features: {len(INAPP_WINDOWS) * len(INAPP_KEYS)}')
print(f'Excluded: purchase_engagement (leakage), totalEventCount (replaced by adjusted)')

In [ ]:
# Section 1: Churn rates at key time points
print('=== Section 1: Golden Time — Churn Rates ===')
print(f'Total analysis users: {len(df):,}\n')

# Key churn rates
churn_cols = {
    'M10': 'IS_M10_CHURN',
    'M30': 'IS_M30_CHURN',
    'M60 (=H1)': 'IS_M60_CHURN',
    'M360 (=H6)': 'IS_M360_CHURN',
    'D1 (=H24)': 'IS_D1_CHURN',
    'D7': 'IS_D7_CHURN',
    'D30': 'IS_D30_CHURN',
}

print(f'{"Time":<15} {"Churn Rate":>12} {"Churned Users":>15} {"Framing Value":>15}')
print('-' * 60)
framing_values = {'M10': 24.9, 'M360 (=H6)': 44.1, 'D1 (=H24)': 50.3}
for label, col in churn_cols.items():
    rate = df[col].mean() * 100
    count = df[col].sum()
    fv = framing_values.get(label, '')
    fv_str = f'{fv}%' if fv else ''
    print(f'{label:<15} {rate:>11.1f}% {count:>14,} {fv_str:>15}')

In [ ]:
# Channel-wise purchase and churn rates
print('=== Channel-wise Purchase and Churn Rates ===')

def get_channel(row):
    if row['last_touch_is_sa'] == 1:
        return 'SA'
    elif row['last_touch_is_da'] == 1:
        return 'DA'
    elif row['has_touchpoint'] == 0:
        return 'Organic'
    else:
        return 'Other'

df['channel'] = df.apply(get_channel, axis=1)

print(f'\n{"Channel":<10} {"N":>8} {"D7 Purchase":>12} {"D7 Churn":>10}')
print('-' * 45)
for ch in ['SA', 'DA', 'Organic', 'Other']:
    sub = df[df['channel'] == ch]
    if len(sub) > 0:
        p = sub['IS_D7_PURCHASE'].mean() * 100
        c = sub['IS_D7_CHURN'].mean() * 100
        print(f'{ch:<10} {len(sub):>8,} {p:>11.1f}% {c:>9.1f}%')

print(f'\nFraming expected: SA=18.6%/43.0%, DA=11.2%/51.2%, Organic=13.4%/43.6%')

In [ ]:
# M10 churn irreversibility
m10_churned = df[df['IS_M10_CHURN'] == 1]
m10_active = df[df['IS_M10_CHURN'] == 0]

print('=== M10 Churn Irreversibility ===')
print(f'M10 churned users: {len(m10_churned):,} ({len(m10_churned)/len(df)*100:.1f}%)')
print(f'M10 churned D30 purchase rate: {m10_churned["IS_D30_PURCHASE"].mean()*100:.1f}%')
print(f'M10 active D30 purchase rate: {m10_active["IS_D30_PURCHASE"].mean()*100:.1f}%')
print(f'Ratio (active/churned): {m10_active["IS_D30_PURCHASE"].mean()/m10_churned["IS_D30_PURCHASE"].mean():.1f}x')
print(f'\nFraming expected: M10 churned purchase = 2.9%, active = 26.3%, ratio ~1/9')

# M10 churn as fraction of D30 churn
d30_churn_total = df['IS_D30_CHURN'].sum()
m10_as_fraction = len(m10_churned) / d30_churn_total * 100
print(f'\nM10 churn as % of D30 churn: {m10_as_fraction:.1f}% (framing: ~37%)')

### Section 2: UA Predictive Power

In [ ]:
# Prepare features for modeling
# === Feature Definition ===
# 77 UA features = all columns MINUS identifiers, device, targets, inapp, ltv, creative image, media_type

DEVICE_FEATURES = ['OS_NAME', 'DEVICE_MANUFACTURER', 'DEVICE_LANGUAGE',
                   'DEVICE_TIMEZONE', 'DEVICE_OSVERSION', 'DEVICE_CARRIER']

# Exclusion-based UA feature definition (ensures we get exactly 77)
exclude_prefixes = ['IS_', 'TARGET_', 'inapp_', 'ltv']
meta_cols = ['IDFA', 'IDFV', 'GAID', 'INSTALL_TIMESTAMP', 'USER_ID']
creative_image_cols = [
    'creative_brightness_mean', 'creative_saturation_mean', 'creative_hue_mean',
    'creative_brightness_std', 'creative_saturation_std', 'creative_hue_std',
    'creative_colorfulness', 'creative_symmetry_score',
    'brightness_mean', 'saturation_mean', 'color_entropy', 'edge_density',
    'hue_cos', 'hue_sin', 'symmetry_score', 'vertical_symmetry_score'
]
exclude_exact = set(meta_cols + DEVICE_FEATURES + creative_image_cols + ['media_type', 'keyword_list', 'ocr_text'])

# Use original CSV columns to avoid including derived columns
original_cols = pd.read_csv('coldstart_dataset_260304.csv', nrows=0).columns.tolist()
ua_cols = []
for col in original_cols:
    if any(col.startswith(p) for p in exclude_prefixes):
        continue
    if col in exclude_exact:
        continue
    ua_cols.append(col)

print(f'UA features: {len(ua_cols)}')
assert len(ua_cols) == 77, f'Expected 77 UA features, got {len(ua_cols)}'
print('UA feature list:')
for i, c in enumerate(ua_cols):
    print(f'  {i+1:2d}. {c}')

# Creative text features (subset of 77 UA features)
creative_text_in_ua = [c for c in ua_cols if c.startswith(('llm_', 'rule_', 'has_ocr', 'has_any_creative', 'has_usable_creative', 'has_broken_image'))]
ua_no_creative = [c for c in ua_cols if c not in creative_text_in_ua]
print(f'\nCreative-related features (in UA): {len(creative_text_in_ua)}')
print(f'UA without creative: {len(ua_no_creative)}')

# Device encoding: pd.get_dummies (as per feature_spec.md)
for col in DEVICE_FEATURES:
    df[col] = df[col].fillna('unknown')

df_encoded = df.copy()
device_dummies = pd.get_dummies(df_encoded[DEVICE_FEATURES], dtype=int)
device_cols = device_dummies.columns.tolist()
df_encoded = pd.concat([df_encoded, device_dummies], axis=1)

df_encoded[ua_cols] = df_encoded[ua_cols].apply(pd.to_numeric, errors='coerce')
df_encoded[ua_cols] = df_encoded[ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

# Sample for modeling (50K for speed; full-data results verified separately)
N_SAMPLE = 50_000
np.random.seed(42)
sample_idx = np.random.choice(len(df_encoded), N_SAMPLE, replace=False)
df_sample = df_encoded.iloc[sample_idx].reset_index(drop=True)

print(f'\nDevice features (one-hot dummies): {len(device_cols)}')
print(f'Sample size: {N_SAMPLE:,}')
print(f'\nNote: Final numbers in research_framing_kor.md are from full 385,025 data.')

In [ ]:
# Cross-validated AUC computation utility
def compute_cv_auc(X, y, model_type='lr', n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    aucs = []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        if model_type == 'lr':
            model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        elif model_type == 'rf':
            model = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        y_prob = model.predict_proba(X_test)[:, 1]
        aucs.append(roc_auc_score(y_test, y_prob))
    return np.mean(aucs), np.std(aucs)

# 4-model AUC comparison (D7 Purchase)
print('=== 4-Model AUC Comparison (D7 Purchase) ===')
target = 'IS_D7_PURCHASE'
y = df_sample[target].values

inapp_m10_cols = [c for c in get_inapp_features('m10') if c in df_sample.columns]

configs = {
    'A: Device Only': device_cols,
    'B: Device + UA': device_cols + ua_cols,
    'C: Device + UA + InApp(M10)': device_cols + ua_cols + inapp_m10_cols,
    'D: Device + InApp(M10)': device_cols + inapp_m10_cols,
}

for algo in ['lr', 'rf']:
    print(f'\nAlgorithm: {algo.upper()}')
    for name, cols_list in configs.items():
        X = df_sample[cols_list].replace([np.inf, -np.inf], np.nan).fillna(0).values
        mean_auc, std_auc = compute_cv_auc(X, y, model_type=algo)
        print(f'  {name}: AUC = {mean_auc:.4f} (±{std_auc:.4f})')

print(f'\nFraming expected: Device=0.545, +UA=0.600, +InApp=0.757, InApp-only=0.748')

In [ ]:
# Top/Bottom 10% analysis — using CV predictions (not in-sample)
# FIX: Was evaluating on training data, now using cross-validated predictions
print('=== Top/Bottom 10% Purchase Rate Gap (CV-based) ===')

from sklearn.model_selection import cross_val_predict

for target_label, target_col in [('M10 Purchase', 'IS_M10_PURCHASE'),
                                  ('M360 Purchase (=H6)', 'IS_M360_PURCHASE'),
                                  ('D7 Purchase', 'IS_D7_PURCHASE')]:
    X_dev_ua = df_sample[device_cols + ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_t = df_sample[target_col].values
    
    # CV predictions (out-of-fold)
    lr = LogisticRegression(max_iter=1000, random_state=42)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    probs = cross_val_predict(lr, X_dev_ua, y_t, cv=skf, method='predict_proba')[:, 1]
    
    top10 = np.percentile(probs, 90)
    bot10 = np.percentile(probs, 10)
    top_mask = probs >= top10
    bot_mask = probs <= bot10
    
    top_rate = y_t[top_mask].mean() * 100
    bot_rate = y_t[bot_mask].mean() * 100
    ratio = top_rate / bot_rate if bot_rate > 0 else float('inf')
    
    print(f'{target_label}: Top10%={top_rate:.1f}%, Bot10%={bot_rate:.1f}%, Gap={ratio:.1f}x')

    # Device only comparison (also CV-based)
    X_dev = df_sample[device_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    lr2 = LogisticRegression(max_iter=1000, random_state=42)
    probs2 = cross_val_predict(lr2, X_dev, y_t, cv=skf, method='predict_proba')[:, 1]
    top10_2 = np.percentile(probs2, 90)
    bot10_2 = np.percentile(probs2, 10)
    top_rate_2 = y_t[probs2 >= top10_2].mean() * 100
    bot_rate_2 = y_t[probs2 <= bot10_2].mean() * 100
    ratio_2 = top_rate_2 / bot_rate_2 if bot_rate_2 > 0 else float('inf')
    print(f'  (Device only): Top10%={top_rate_2:.1f}%, Bot10%={bot_rate_2:.1f}%, Gap={ratio_2:.1f}x')

print(f'\nNote: Using cross-validated (out-of-fold) predictions to avoid overfitting bias')

In [ ]:
# Feature importance (RF) and standardized coefficients (LR)
print('=== Feature Importance Analysis ===')

X_ua = df_sample[ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
y_d7 = df_sample['IS_D7_PURCHASE'].values

# LR standardized coefficients
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_ua)
lr_std = LogisticRegression(max_iter=1000, random_state=42)
lr_std.fit(X_scaled, y_d7)
lr_coefs = pd.Series(lr_std.coef_[0], index=ua_cols).sort_values()

print('\nLR Standardized Coefficients (Top 5 positive):')
for name, val in lr_coefs.tail(5).items():
    print(f'  {name}: {val:+.3f}')
print('\nLR Standardized Coefficients (Top 5 negative):')
for name, val in lr_coefs.head(5).items():
    print(f'  {name}: {val:+.3f}')

# RF feature importance
rf_imp = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_imp.fit(X_ua, y_d7)
rf_importance = pd.Series(rf_imp.feature_importances_, index=ua_cols).sort_values(ascending=False)
print('\nRF Feature Importance (Top 10):')
for name, val in rf_importance.head(10).items():
    print(f'  {name}: {val:.4f}')

print(f'\nFraming expected: latency(-0.323), SA_count(+0.113)')

In [ ]:
# Feature importance decay over time (UA % contribution)
# FIX: Removed d7 window — using inapp_d7 to predict IS_D7_PURCHASE is temporal leakage
print('=== Feature Importance Decay Over Time ===')

decay_windows = ['m10', 'm30', 'm60', 'm360', 'd1']  # d7 removed (temporal leakage with D7 target)
target = 'IS_D7_PURCHASE'
y_d7 = df_sample[target].values

print(f'{"Window":<10} {"Device %":>10} {"UA %":>10} {"InApp %":>10}')
print('-' * 45)

for window in decay_windows:
    inapp_w_cols = [c for c in get_inapp_features(window) if c in df_sample.columns]
    all_cols = device_cols + ua_cols + inapp_w_cols
    X_all = df_sample[all_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_all, y_d7)
    importance = pd.Series(rf.feature_importances_, index=all_cols)
    
    device_pct = importance[device_cols].sum() / importance.sum() * 100
    ua_pct = importance[ua_cols].sum() / importance.sum() * 100
    inapp_pct = importance[inapp_w_cols].sum() / importance.sum() * 100
    
    print(f'{window:<10} {device_pct:>9.1f}% {ua_pct:>9.1f}% {inapp_pct:>9.1f}%')

print(f'\nNote: d7 window removed to prevent temporal leakage (inapp_d7 overlaps IS_D7_PURCHASE)')

In [ ]:
# AUC decay: incremental UA value over time
print('=== AUC Decay: Incremental UA Value ===')
target = 'IS_D30_PURCHASE'
y_d30 = df_sample[target].values

print(f'{"Window":<10} {"AUC (no UA)":>12} {"AUC (with UA)":>14} {"ΔUA":>8}')
print('-' * 48)

decay_results = []
for window in INAPP_WINDOWS:
    inapp_cols_w = [c for c in get_inapp_features(window) if c in df_sample.columns]
    
    X_no_ua = df_sample[device_cols + inapp_cols_w].replace([np.inf, -np.inf], np.nan).fillna(0).values
    X_with_ua = df_sample[device_cols + ua_cols + inapp_cols_w].replace([np.inf, -np.inf], np.nan).fillna(0).values
    
    auc_no, _ = compute_cv_auc(X_no_ua, y_d30, 'lr')
    auc_with, _ = compute_cv_auc(X_with_ua, y_d30, 'lr')
    delta = auc_with - auc_no
    
    decay_results.append({'window': window, 'AUC_no_UA': auc_no, 'AUC_with_UA': auc_with, 'delta_UA': delta})
    print(f'{window:<10} {auc_no:>11.4f} {auc_with:>13.4f} {delta:>+7.4f}')

decay_df = pd.DataFrame(decay_results)
print(f'\nFraming expected: m10=+0.0106, m30=+0.0087, m60(H1)=+0.0081, m360(H6)=+0.0052, d1(H24)=+0.0021')

In [ ]:
# Latency analysis (inverted-U)
print('=== Latency Analysis ===')
paid = df[df['has_touchpoint'] == 1].copy()

paid['latency_hours'] = paid['latency'] / 3600
bins = [0, 1, 15, 23, 24, float('inf')]
labels = ['~1h', '1~15h', '15~23h', '23~24h', '24h+']
paid['latency_bin'] = pd.cut(paid['latency_hours'], bins=bins, labels=labels, right=False)

print(f'{"Latency":<12} {"N":>8} {"D7 Churn":>10} {"D7 Purchase":>12}')
print('-' * 45)
for label in labels:
    sub = paid[paid['latency_bin'] == label]
    if len(sub) > 0:
        churn = sub['IS_D7_CHURN'].mean() * 100
        purch = sub['IS_D7_PURCHASE'].mean() * 100
        print(f'{label:<12} {len(sub):>8,} {churn:>9.1f}% {purch:>11.1f}%')

print(f'\nFraming expected: ~1h=51.6%/12.8%, 23-24h=42.7%/16.2%')

In [ ]:
# OS × Channel cross-effect
print('=== OS × Channel Cross-Effect ===')

print(f'{"OS × Channel":<20} {"N":>8} {"D7 Churn":>10} {"D7 Purchase":>12}')
print('-' * 55)
for os in ['Android', 'iOS']:
    for ch in ['DA', 'SA', 'Organic']:
        mask = (df['OS_NAME'] == os) & (df['channel'] == ch)
        sub = df[mask]
        if len(sub) > 0:
            churn = sub['IS_D7_CHURN'].mean() * 100
            purch = sub['IS_D7_PURCHASE'].mean() * 100
            print(f'{os} × {ch:<10} {len(sub):>8,} {churn:>9.1f}% {purch:>11.1f}%')

print(f'\nFraming expected: Android×DA=9.4%/54.7%, iOS×SA=25.0%/38.5%')

In [ ]:
# Bootstrap CI for UA lift — using OOB evaluation
# FIX: Was evaluating on in-bag (training) data, now using out-of-bag samples
print('=== Bootstrap CI for UA AUC Lift (OOB evaluation) ===')

n_bootstrap = 200
lifts = []
y_d7 = df_sample['IS_D7_PURCHASE'].values
N = len(df_sample)

for i in range(n_bootstrap):
    idx = np.random.choice(N, N, replace=True)
    oob = np.setdiff1d(np.arange(N), np.unique(idx))
    
    if len(oob) < 100:  # skip if too few OOB samples
        continue
    
    # Fit on bootstrap sample
    X_dev_train = df_sample.iloc[idx][device_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    X_devua_train = df_sample.iloc[idx][device_cols + ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_train = y_d7[idx]
    
    # Evaluate on OOB
    X_dev_oob = df_sample.iloc[oob][device_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    X_devua_oob = df_sample.iloc[oob][device_cols + ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_oob = y_d7[oob]
    
    lr_dev = LogisticRegression(max_iter=1000, random_state=42)
    lr_dev.fit(X_dev_train, y_train)
    auc_dev = roc_auc_score(y_oob, lr_dev.predict_proba(X_dev_oob)[:, 1])
    
    lr_ua = LogisticRegression(max_iter=1000, random_state=42)
    lr_ua.fit(X_devua_train, y_train)
    auc_ua = roc_auc_score(y_oob, lr_ua.predict_proba(X_devua_oob)[:, 1])
    
    lifts.append(auc_ua - auc_dev)

lifts = np.array(lifts)
ci_low = np.percentile(lifts, 2.5)
ci_high = np.percentile(lifts, 97.5)
print(f'Valid bootstrap iterations: {len(lifts)}')
print(f'Mean UA lift: {lifts.mean():.4f}')
print(f'95% CI: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'P(lift > 0): {(lifts > 0).mean()*100:.1f}%')
print(f'\nNote: OOB evaluation (~36.8% of samples held out per iteration)')

In [ ]:
# Paid vs Organic prediction gap
print('=== Paid vs Organic Prediction Gap ===')

paid_mask = df_sample['has_touchpoint'] == 1
organic_mask = df_sample['has_touchpoint'] == 0

for window in ['m10', 'm360', 'd7']:
    inapp_w = [c for c in get_inapp_features(window) if c in df_sample.columns]
    all_feat = device_cols + ua_cols + inapp_w
    
    # Paid
    paid_df = df_sample[paid_mask]
    X_p = paid_df[all_feat].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_p = paid_df['IS_D30_PURCHASE'].values
    auc_p, _ = compute_cv_auc(X_p, y_p, 'lr')
    
    # Organic
    org_df = df_sample[organic_mask]
    X_o = org_df[all_feat].replace([np.inf, -np.inf], np.nan).fillna(0).values
    y_o = org_df['IS_D30_PURCHASE'].values
    auc_o, _ = compute_cv_auc(X_o, y_o, 'lr')
    
    print(f'{window}: Paid AUC={auc_p:.3f}, Organic AUC={auc_o:.3f}, Gap={auc_p-auc_o:+.3f}')

### Section 3: RL Framework Numbers

In [ ]:
# 74.4% of D30 purchases happen within D7
d30_purchasers = df[df['IS_D30_PURCHASE'] == 1]
d7_of_d30 = d30_purchasers['IS_D7_PURCHASE'].mean() * 100
print(f'D30 purchasers who purchased within D7: {d7_of_d30:.1f}% (framing: 74.4%)')

# D7 non-purchasers who convert by D30
d7_non = df[df['IS_D7_PURCHASE'] == 0]
d7_to_d30 = d7_non['IS_D30_PURCHASE'].mean() * 100
print(f'D7 non-purchasers who convert by D30: {d7_to_d30:.1f}% (framing: 6.3%)')

In [ ]:
# Paid vs Organic purchase/churn comparison
print('=== Paid vs Organic Comparison ===')
paid_users = df[df['has_touchpoint'] == 1]
organic_users = df[df['has_touchpoint'] == 0]

metrics = [
    ('D7 Purchase', 'IS_D7_PURCHASE'),
    ('D7 Churn', 'IS_D7_CHURN'),
    ('M10 Churn', 'IS_M10_CHURN'),
    ('D30 Churn', 'IS_D30_CHURN'),
]

print(f'{"Metric":<15} {"Paid":>8} {"Organic":>10} {"Diff":>8}')
print('-' * 45)
for label, col in metrics:
    p = paid_users[col].mean() * 100
    o = organic_users[col].mean() * 100
    print(f'{label:<15} {p:>7.1f}% {o:>9.1f}% {p-o:>+7.1f}%p')

print(f'\nFraming: Paid D7 Purchase=15.8%, Organic=13.4%, Paid M10 Churn=25.1%, Organic=19.2%')

### Appendix Numbers Verification

In [ ]:
# Keyword intent purchase rates
print('=== Keyword Intent Purchase Rates ===')

kw_types = [
    ('Brand Search', 'kw_brand_search'),
    ('Product Search', 'kw_product_search'),
    ('Promo Search', 'kw_promo_season_search'),
]

no_kw = df[df['has_term'] == 0]
no_kw_rate = no_kw['IS_D7_PURCHASE'].mean() * 100
print(f'No keyword users: {len(no_kw):,}, D7 purchase: {no_kw_rate:.1f}%')

for label, col in kw_types:
    sub = df[df[col] == 1]
    if len(sub) > 0:
        rate = sub['IS_D7_PURCHASE'].mean() * 100
        print(f'{label}: N={len(sub):,}, D7 purchase={rate:.1f}%, vs no-kw={rate/no_kw_rate:.2f}x')

In [ ]:
# Touch count quintile analysis
print('=== Touch Count Analysis ===')
paid = df[df['has_touchpoint'] == 1].copy()

bins = [0, 1, 3, 10, 30, float('inf')]
labels = ['1', '2~3', '4~10', '11~30', '31+']
paid['touch_bin'] = pd.cut(paid['total_touch_count'], bins=bins, labels=labels, right=True)

print(f'{"Touches":<10} {"N":>8} {"Pct":>6} {"D7 Purchase":>12} {"D7 Churn":>10}')
print('-' * 50)
for label in labels:
    sub = paid[paid['touch_bin'] == label]
    pct = len(sub) / len(paid) * 100
    purch = sub['IS_D7_PURCHASE'].mean() * 100
    churn = sub['IS_D7_CHURN'].mean() * 100
    print(f'{label:<10} {len(sub):>8,} {pct:>5.0f}% {purch:>11.1f}% {churn:>9.1f}%')

In [ ]:
# Multi-channel vs single-channel
print('=== Multi-channel vs Single-channel ===')
paid = df[df['has_touchpoint'] == 1].copy()

single = paid[paid['unique_channel_count'] == 1]
multi = paid[paid['unique_channel_count'] >= 2]

print(f'{"Type":<15} {"N":>8} {"D7 Purchase":>12} {"D7 Churn":>10}')
print('-' * 50)
print(f'{"Single Ch":<15} {len(single):>8,} {single["IS_D7_PURCHASE"].mean()*100:>11.1f}% {single["IS_D7_CHURN"].mean()*100:>9.1f}%')
print(f'{"Multi Ch (2+)":<15} {len(multi):>8,} {multi["IS_D7_PURCHASE"].mean()*100:>11.1f}% {multi["IS_D7_CHURN"].mean()*100:>9.1f}%')
print(f'\nFraming: Single=15.9%/46.2%, Multi=15.7%/46.8%')

In [ ]:
# Fraud robustness check — Full data, pd.get_dummies, RF (same methodology as main analysis)
print('=== Fraud Robustness (Full Data, pd.get_dummies, RF) ===')

rf_params = dict(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for label, include_fraud in [('Fraud excluded (385K)', False), ('Fraud included (438K)', True)]:
    if include_fraud:
        subset = df_new.copy()
    else:
        subset = df.copy()
    
    print(f'\n--- {label}: {len(subset):,} users ---')
    
    # Device encoding (pd.get_dummies, same as main analysis)
    device_df = pd.get_dummies(subset[DEVICE_FEATURES].fillna('unknown'))
    
    # UA features
    ua_df = subset[ua_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    
    y = subset['IS_D7_PURCHASE'].values
    
    # Model A: Device only
    X_A = device_df.values
    proba_A = cross_val_predict(RandomForestClassifier(**rf_params), X_A, y, cv=cv, method='predict_proba')[:, 1]
    auc_A = roc_auc_score(y, proba_A)
    
    # Model B: Device + UA
    X_B = np.hstack([device_df.values, ua_df.values])
    proba_B = cross_val_predict(RandomForestClassifier(**rf_params), X_B, y, cv=cv, method='predict_proba')[:, 1]
    auc_B = roc_auc_score(y, proba_B)
    
    lift = auc_B - auc_A
    print(f'  Model A (Device): AUC = {auc_A:.4f}')
    print(f'  Model B (Device+UA): AUC = {auc_B:.4f}')
    print(f'  UA Lift: {lift:+.4f}')
    
    # Top/Bottom 10%
    n = len(y)
    top10 = np.argsort(proba_B)[-n//10:]
    bot10 = np.argsort(proba_B)[:n//10]
    top_rate = y[top10].mean() * 100
    bot_rate = y[bot10].mean() * 100
    ratio = top_rate / bot_rate if bot_rate > 0 else float('inf')
    print(f'  Top 10%: {top_rate:.1f}%, Bottom 10%: {bot_rate:.1f}%, Ratio: {ratio:.1f}x')

print('\n✓ Conclusion: UA lift is robust to fraud inclusion (+0.067 → +0.068)')
print('  Top/Bot ratio decreases (9.8x → 7.8x) due to fraud noise diluting extremes')

---
## Phase 3: 30-Minute Resolution Time Analysis

In [ ]:
# Cumulative churn curve at 30-minute resolution
print('=== 30-Minute Resolution Churn Curve ===')

churn_windows_30m = ['m10', 'm30', 'm60', 'm90', 'm120', 'm150', 'm180',
                      'm210', 'm240', 'm270', 'm300', 'm330', 'm360',
                      'd1', 'd2', 'd3', 'd7', 'd14', 'd30']

churn_rates = []
purchase_rates = []
time_labels = []

for w in churn_windows_30m:
    churn_col = f'IS_{w.upper()}_CHURN'
    purch_col = f'IS_{w.upper()}_PURCHASE'
    if churn_col in df.columns:
        cr = df[churn_col].mean() * 100
        pr = df[purch_col].mean() * 100
        churn_rates.append(cr)
        purchase_rates.append(pr)
        time_labels.append(w)

print(f'{"Window":<10} {"Churn %":>10} {"Purchase %":>12} {"Δ Churn":>10} {"Δ Purchase":>12}')
print('-' * 58)
prev_c, prev_p = 0, 0
for i, (w, c, p) in enumerate(zip(time_labels, churn_rates, purchase_rates)):
    dc = c - prev_c
    dp = p - prev_p
    print(f'{w:<10} {c:>9.1f}% {p:>11.1f}% {dc:>+9.1f}%p {dp:>+11.1f}%p')
    prev_c, prev_p = c, p

In [ ]:
# UA AUC decay at 30-minute resolution (19 time points)
print('=== UA AUC Decay at 30-Minute Resolution ===')
target = 'IS_D30_PURCHASE'
y_d30 = df_sample[target].values

all_windows = ['m10', 'm30', 'm60', 'm90', 'm120', 'm150', 'm180',
               'm210', 'm240', 'm270', 'm300', 'm330', 'm360',
               'd1', 'd2', 'd3', 'd7', 'd14', 'd30']

print(f'{"Window":<10} {"AUC (no UA)":>12} {"AUC (+UA)":>12} {"ΔUA":>8} {"InApp #":>8}')
print('-' * 55)

fine_decay = []
for window in all_windows:
    inapp_cols_w = [c for c in get_inapp_features(window) if c in df_sample.columns]
    
    X_no = df_sample[device_cols + inapp_cols_w].replace([np.inf, -np.inf], np.nan).fillna(0).values
    X_with = df_sample[device_cols + ua_cols + inapp_cols_w].replace([np.inf, -np.inf], np.nan).fillna(0).values
    
    auc_no, _ = compute_cv_auc(X_no, y_d30, 'lr')
    auc_with, _ = compute_cv_auc(X_with, y_d30, 'lr')
    delta = auc_with - auc_no
    
    fine_decay.append({'window': window, 'AUC_no_UA': auc_no, 'AUC_with_UA': auc_with, 
                       'delta_UA': delta, 'n_inapp': len(inapp_cols_w)})
    print(f'{window:<10} {auc_no:>11.4f} {auc_with:>11.4f} {delta:>+7.4f} {len(inapp_cols_w):>8}')

fine_decay_df = pd.DataFrame(fine_decay)

In [ ]:
# Feature importance transition at 30-minute resolution
# FIX: Removed d7 from fi_windows — temporal leakage with IS_D7_PURCHASE target
print('=== Feature Importance Transition (30-min Resolution) ===')

fi_windows = ['m10', 'm30', 'm60', 'm90', 'm120', 'm150', 'm180',
              'm210', 'm240', 'm270', 'm300', 'm330', 'm360', 'd1']  # d7 removed

fi_results = []
y_d7 = df_sample['IS_D7_PURCHASE'].values

print(f'{"Window":<10} {"Device %":>10} {"UA %":>10} {"InApp %":>10}')
print('-' * 45)

for window in fi_windows:
    inapp_w_cols = [c for c in get_inapp_features(window) if c in df_sample.columns]
    all_cols_w = device_cols + ua_cols + inapp_w_cols
    X_all = df_sample[all_cols_w].replace([np.inf, -np.inf], np.nan).fillna(0).values
    
    rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_all, y_d7)
    importance = pd.Series(rf.feature_importances_, index=all_cols_w)
    
    d_pct = importance[device_cols].sum() / importance.sum() * 100
    u_pct = importance[ua_cols].sum() / importance.sum() * 100
    i_pct = importance[inapp_w_cols].sum() / importance.sum() * 100
    
    fi_results.append({'window': window, 'device': d_pct, 'ua': u_pct, 'inapp': i_pct})
    print(f'{window:<10} {d_pct:>9.1f}% {u_pct:>9.1f}% {i_pct:>9.1f}%')

fi_df = pd.DataFrame(fi_results)
print(f'\nNote: d7 excluded from feature importance when target is IS_D7_PURCHASE (temporal leakage)')

In [ ]:
# Decay model fitting (exponential vs power law)
from scipy.optimize import curve_fit

print('=== Decay Model Fitting ===')

# Convert windows to minutes
window_minutes = {
    'm10': 10, 'm30': 30, 'm60': 60, 'm90': 90, 'm120': 120, 'm150': 150,
    'm180': 180, 'm210': 210, 'm240': 240, 'm270': 270, 'm300': 300,
    'm330': 330, 'm360': 360, 'd1': 1440, 'd2': 2880, 'd3': 4320,
    'd7': 10080, 'd14': 20160, 'd30': 43200
}

x_data = np.array([window_minutes[w] for w in fine_decay_df['window']])
y_data = fine_decay_df['delta_UA'].values

# Filter positive values for fitting
mask = y_data > 0
x_fit, y_fit = x_data[mask], y_data[mask]

# Exponential: y = a * exp(-b * x)
def exp_decay(x, a, b):
    return a * np.exp(-b * x)

# Power law: y = a * x^(-b)
def power_decay(x, a, b):
    return a * np.power(x, -b)

try:
    popt_exp, _ = curve_fit(exp_decay, x_fit, y_fit, p0=[0.01, 0.001], maxfev=10000)
    y_pred_exp = exp_decay(x_fit, *popt_exp)
    ss_res_exp = np.sum((y_fit - y_pred_exp) ** 2)
    ss_tot = np.sum((y_fit - np.mean(y_fit)) ** 2)
    r2_exp = 1 - ss_res_exp / ss_tot
    print(f'Exponential: a={popt_exp[0]:.4f}, b={popt_exp[1]:.6f}, R²={r2_exp:.4f}')
    print(f'  Half-life: {np.log(2)/popt_exp[1]:.0f} minutes ({np.log(2)/popt_exp[1]/60:.1f} hours)')
except Exception as e:
    print(f'Exponential fit failed: {e}')

try:
    popt_pow, _ = curve_fit(power_decay, x_fit, y_fit, p0=[0.1, 0.5], maxfev=10000)
    y_pred_pow = power_decay(x_fit, *popt_pow)
    ss_res_pow = np.sum((y_fit - y_pred_pow) ** 2)
    r2_pow = 1 - ss_res_pow / ss_tot
    print(f'Power law: a={popt_pow[0]:.4f}, b={popt_pow[1]:.4f}, R²={r2_pow:.4f}')
except Exception as e:
    print(f'Power law fit failed: {e}')

if r2_exp > r2_pow:
    print(f'\n→ Exponential decay fits better (R²={r2_exp:.4f} vs {r2_pow:.4f})')
else:
    print(f'\n→ Power law decay fits better (R²={r2_pow:.4f} vs {r2_exp:.4f})')

In [ ]:
# Information crossover point: when does InApp surpass UA?
print('=== Information Crossover: InApp vs UA ===')

print(f'{"Window":<10} {"UA %":>10} {"InApp %":>10} {"UA > InApp?":>12}')
print('-' * 45)
crossover_found = False
for _, row in fi_df.iterrows():
    dominant = 'Yes' if row['ua'] > row['inapp'] else 'No'
    print(f'{row["window"]:<10} {row["ua"]:>9.1f}% {row["inapp"]:>9.1f}% {dominant:>12}')
    if not crossover_found and row['ua'] < row['inapp']:
        print(f'  → InApp surpasses UA at {row["window"]}')
        crossover_found = True

---
## Phase 4: LTV Analysis

In [ ]:
# Parse LTV and analyze
print('=== LTV Analysis ===')

# Parse LTV JSON
df['ltv_parsed'] = df['ltv'].apply(parse_ltv)

# Check LTV structure from first few non-empty values
non_empty = df[df['ltv_parsed'].apply(len) > 0]['ltv_parsed']
print(f'Users with LTV data: {len(non_empty):,}')
print(f'Users without LTV data: {(df["ltv_parsed"].apply(len) == 0).sum():,}')

if len(non_empty) > 0:
    sample_keys = set()
    for v in non_empty.head(100):
        sample_keys.update(v.keys())
    print(f'\nLTV keys: {sorted(sample_keys)}')
    print(f'\nSample LTV values:')
    for i, v in enumerate(non_empty.head(5)):
        print(f'  {v}')

In [ ]:
# Extract total LTV (sum of all time windows)
def extract_total_ltv(ltv_dict):
    if not ltv_dict:
        return 0.0
    total = 0
    for key, val in ltv_dict.items():
        try:
            total += float(val)
        except (ValueError, TypeError):
            pass
    return total

# Try to understand LTV structure first
# LTV might be total cumulative or per-window
sample_ltvs = non_empty.head(20).tolist()
for ltv in sample_ltvs[:5]:
    print(ltv)

# Extract LTV by the latest/largest time window key
# If it's cumulative, we want the max window's value
# If it's per-window, we want the sum
print('\n--- Attempting LTV extraction ---')
df['ltv_total'] = df['ltv_parsed'].apply(extract_total_ltv)

In [ ]:
# LTV distribution
ltv_nonzero = df[df['ltv_total'] > 0]['ltv_total']
print(f'Users with LTV > 0: {len(ltv_nonzero):,}')
print(f'\nLTV Distribution (among buyers):')
print(ltv_nonzero.describe())
print(f'\nPercentiles:')
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f'  P{p}: {ltv_nonzero.quantile(p/100):,.0f}')

In [ ]:
# Revenue-based UA value: Top/Bottom 10% predicted users' average LTV
print('=== Revenue-based UA Value ===')

X_dev_ua = df_sample[device_cols + ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
y_d7 = df_sample['IS_D7_PURCHASE'].values

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_dev_ua, y_d7)
probs = lr.predict_proba(X_dev_ua)[:, 1]

# Get LTV for sampled users
# Need to merge back with LTV
df_sample_ltv = df_sample.copy()
df_sample_ltv['pred_prob'] = probs

# Match LTV from df
ltv_map = df.set_index('USER_ID')['ltv_total'].to_dict()
df_sample_ltv['ltv_total'] = df_sample_ltv['USER_ID'].map(ltv_map).fillna(0)

top10 = np.percentile(probs, 90)
bot10 = np.percentile(probs, 10)

top_ltv = df_sample_ltv[df_sample_ltv['pred_prob'] >= top10]['ltv_total'].mean()
bot_ltv = df_sample_ltv[df_sample_ltv['pred_prob'] <= bot10]['ltv_total'].mean()
overall_ltv = df_sample_ltv['ltv_total'].mean()

print(f'Top 10% predicted users avg LTV: {top_ltv:,.0f}')
print(f'Bottom 10% predicted users avg LTV: {bot_ltv:,.0f}')
print(f'Overall avg LTV: {overall_ltv:,.0f}')
if bot_ltv > 0:
    print(f'LTV ratio (Top/Bottom): {top_ltv/bot_ltv:.1f}x')

In [ ]:
# LTV by channel
print('=== LTV by Channel ===')

for ch in ['SA', 'DA', 'Organic']:
    sub = df[df['channel'] == ch]
    mean_ltv = sub['ltv_total'].mean()
    median_ltv = sub[sub['ltv_total'] > 0]['ltv_total'].median() if (sub['ltv_total'] > 0).any() else 0
    purch_ltv = sub[sub['ltv_total'] > 0]['ltv_total'].mean() if (sub['ltv_total'] > 0).any() else 0
    print(f'{ch}: Mean LTV (all)={mean_ltv:,.0f}, Mean LTV (buyers)={purch_ltv:,.0f}, Median (buyers)={median_ltv:,.0f}')

In [ ]:
# LTV by latency quintile
print('=== LTV by Latency ===')
paid = df[df['has_touchpoint'] == 1].copy()
paid['latency_hours'] = paid['latency'] / 3600

bins = [0, 1, 15, 23, 24, float('inf')]
labels = ['~1h', '1~15h', '15~23h', '23~24h', '24h+']
paid['latency_bin'] = pd.cut(paid['latency_hours'], bins=bins, labels=labels, right=False)

print(f'{"Latency":<12} {"N":>8} {"Mean LTV (all)":>15} {"Mean LTV (buyers)":>18}')
print('-' * 58)
for label in labels:
    sub = paid[paid['latency_bin'] == label]
    mean_all = sub['ltv_total'].mean()
    buyers = sub[sub['ltv_total'] > 0]
    mean_buyers = buyers['ltv_total'].mean() if len(buyers) > 0 else 0
    print(f'{label:<12} {len(sub):>8,} {mean_all:>14,.0f} {mean_buyers:>17,.0f}')

---
## Phase 5: Creative Analysis Verification & Integration

In [ ]:
# Creative data overview
print('=== Creative Data Overview ===')

# Google/Meta touchpoint
has_gm = df['has_gm_touchpoint'].sum()
has_creative = df['has_any_creative_url'].sum() if 'has_any_creative_url' in df.columns else 0
has_usable = df['has_usable_creative'].sum() if 'has_usable_creative' in df.columns else 0
has_broken = df['has_broken_image'].sum() if 'has_broken_image' in df.columns else 0

print(f'Google/Meta touchpoint users: {has_gm:,} ({has_gm/len(df)*100:.1f}%)')
print(f'Has any creative URL: {has_creative:,} ({has_creative/len(df)*100:.1f}%)')
print(f'Has usable creative: {has_usable:,} ({has_usable/len(df)*100:.1f}%)')
print(f'Has broken image (catalog ads): {has_broken:,} ({has_broken/len(df)*100:.1f}%)')

# DA users
da_users = df[df['last_touch_is_da'] == 1]
print(f'\nDA (last touch) users: {len(da_users):,} ({len(da_users)/len(df)*100:.1f}%)')

In [ ]:
# Creative text message types purchase rates
print('=== Creative Text Message Types ===')

llm_cols = [c for c in df.columns if c.startswith('llm_')]
has_ocr = df[df['has_ocr_text'] == 1] if 'has_ocr_text' in df.columns else pd.DataFrame()

if len(has_ocr) > 0:
    print(f'Users with OCR text: {len(has_ocr):,}')
    baseline = has_ocr['IS_D7_PURCHASE'].mean() * 100
    print(f'Baseline purchase rate (all OCR users): {baseline:.1f}%\n')
    
    print(f'{"Message Type":<25} {"N":>8} {"D7 Purchase":>12} {"Diff":>8}')
    print('-' * 58)
    for col in llm_cols:
        sub = has_ocr[has_ocr[col] == 1]
        if len(sub) > 0:
            rate = sub['IS_D7_PURCHASE'].mean() * 100
            diff = rate - baseline
            print(f'{col:<25} {len(sub):>8,} {rate:>11.1f}% {diff:>+7.1f}%p')

    print(f'\nFraming expected: brand_trust +2.4%p, coupon (reward_benefit) -2.8%p')

In [ ]:
# Creative AUC analysis
# Creative text features are ALREADY part of 77 UA features
# To measure creative text contribution, compare UA(no creative) vs UA(with creative)
print('=== Creative AUC Analysis ===')

creative_image_features = [c for c in df.columns if c.startswith(('creative_brightness', 'creative_saturation',
    'creative_hue', 'creative_colorfulness', 'creative_symmetry'))] + \
    ['brightness_mean', 'saturation_mean', 'color_entropy', 'edge_density',
     'hue_cos', 'hue_sin', 'symmetry_score', 'vertical_symmetry_score']

print(f'Creative text features (in UA): {len(creative_text_in_ua)}')
print(f'  {creative_text_in_ua}')
print(f'Creative image features (excluded from UA): {len(creative_image_features)}')
print(f'UA without creative text: {len(ua_no_creative)}')

y_d7 = df_sample['IS_D7_PURCHASE'].values

# Model B: Device + ALL 77 UA (includes creative text)
X_full = df_sample[device_cols + ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
auc_full, _ = compute_cv_auc(X_full, y_d7, 'rf')

# Device + UA without creative text
X_no_creative = df_sample[device_cols + ua_no_creative].replace([np.inf, -np.inf], np.nan).fillna(0).values
auc_no_creative, _ = compute_cv_auc(X_no_creative, y_d7, 'rf')

# Device + UA + creative image (excluded from standard model)
X_with_image = df_sample[device_cols + ua_cols + [c for c in creative_image_features if c in df_sample.columns]].replace([np.inf, -np.inf], np.nan).fillna(0).values
auc_with_image, _ = compute_cv_auc(X_with_image, y_d7, 'rf')

print(f'\nDevice+UA(no creative text) AUC: {auc_no_creative:.4f}')
print(f'Device+UA(with creative text) AUC: {auc_full:.4f}')
print(f'Creative text contribution: {auc_full-auc_no_creative:+.4f}')
print(f'Device+UA+Image AUC: {auc_with_image:.4f}')
print(f'Creative image contribution: {auc_with_image-auc_full:+.4f}')
print(f'\nFraming states: Text +0.004, Image +0.001')

In [ ]:
# Device info purchase rates (for Section 1 table)
print('=== Device Info Purchase Rates ===')

# OS
for os in ['Android', 'iOS']:
    sub = df[df['OS_NAME'] == os]
    print(f'{os}: N={len(sub):,}, D7 Purchase={sub["IS_D7_PURCHASE"].mean()*100:.1f}%')

# Manufacturer
for mfr in ['Samsung', 'Apple']:
    sub = df[df['DEVICE_MANUFACTURER'] == mfr]
    print(f'{mfr}: N={len(sub):,}, D7 Purchase={sub["IS_D7_PURCHASE"].mean()*100:.1f}%')

# Language
ko = df[df['DEVICE_LANGUAGE'] == 'ko']
en = df[df['DEVICE_LANGUAGE'] == 'en']
print(f'Domestic (ko): N={len(ko):,} ({len(ko)/len(df)*100:.1f}%), D7 Purchase={ko["IS_D7_PURCHASE"].mean()*100:.1f}%')
print(f'English: N={len(en):,}, D7 Purchase={en["IS_D7_PURCHASE"].mean()*100:.1f}%')

# Install time
time_cols = ['is_installed_02_06', 'is_installed_06_10', 'is_installed_10_14',
             'is_installed_14_18', 'is_installed_18_22', 'is_installed_22_02']
for tc in time_cols:
    sub = df[df[tc] == 1]
    print(f'{tc}: N={len(sub):,}, D7 Purchase={sub["IS_D7_PURCHASE"].mean()*100:.1f}%')

In [ ]:
# PSM analysis: 10-minute activity effect on purchase
# Full propensity score matching implementation
print('=== PSM: First 10-minute Activity Effect ===')
from sklearn.neighbors import NearestNeighbors
from scipy.stats import chi2_contingency

# M10 survivors only
survived = df[df['IS_M10_CHURN'] == 0].copy()
print(f'M10 survivors: {len(survived):,}')

# High/Low activity split (median of core_engagement)
median_ce = survived['inapp_m10_core_engagement'].median()
survived['high_activity'] = (survived['inapp_m10_core_engagement'] > median_ce).astype(int)

high = survived[survived['high_activity'] == 1]
low = survived[survived['high_activity'] == 0]
print(f'High activity: {len(high):,}, D7 Purchase: {high["IS_D7_PURCHASE"].mean()*100:.1f}%')
print(f'Low activity: {len(low):,}, D7 Purchase: {low["IS_D7_PURCHASE"].mean()*100:.1f}%')
naive_gap = (high["IS_D7_PURCHASE"].mean() - low["IS_D7_PURCHASE"].mean()) * 100
print(f'Naive gap: {naive_gap:+.1f}%p')

# Covariates: device (encoded) + 77 UA features
psm_covariates = device_cols + ua_cols
X_psm = survived[psm_covariates].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0).values
treatment = survived['high_activity'].values

# Fit propensity score
scaler_psm = StandardScaler()
X_psm_scaled = scaler_psm.fit_transform(X_psm)
lr_psm = LogisticRegression(max_iter=1000, random_state=42)
lr_psm.fit(X_psm_scaled, treatment)
ps = lr_psm.predict_proba(X_psm_scaled)[:, 1]

print(f'\nPropensity score range: [{ps.min():.4f}, {ps.max():.4f}]')

# 1:1 Nearest neighbor matching (without replacement, caliper 0.05)
treated_idx = np.where(treatment == 1)[0]
control_idx = np.where(treatment == 0)[0]

nn = NearestNeighbors(n_neighbors=1, metric='euclidean')
nn.fit(ps[control_idx].reshape(-1, 1))
distances, indices = nn.kneighbors(ps[treated_idx].reshape(-1, 1))

caliper = 0.05
matched_pairs = []
used_controls = set()
for i, (dist, idx) in enumerate(zip(distances.flatten(), indices.flatten())):
    if dist <= caliper and control_idx[idx] not in used_controls:
        matched_pairs.append((treated_idx[i], control_idx[idx]))
        used_controls.add(control_idx[idx])

print(f'Matched pairs: {len(matched_pairs):,}')

# Outcomes
treated_outcomes = survived.iloc[[p[0] for p in matched_pairs]]['IS_D7_PURCHASE'].values
control_outcomes = survived.iloc[[p[1] for p in matched_pairs]]['IS_D7_PURCHASE'].values

psm_high = treated_outcomes.mean() * 100
psm_low = control_outcomes.mean() * 100
psm_gap = psm_high - psm_low
print(f'\nPost-matching: High={psm_high:.1f}%, Low={psm_low:.1f}%, Gap={psm_gap:+.1f}%p')
print(f'Retention: {psm_gap/naive_gap*100:.0f}% of naive gap')

# Chi-square
contingency = np.array([
    [treated_outcomes.sum(), len(treated_outcomes) - treated_outcomes.sum()],
    [control_outcomes.sum(), len(control_outcomes) - control_outcomes.sum()]
])
chi2, p_val, _, _ = chi2_contingency(contingency, correction=False)
print(f'χ² = {chi2:.1f}, p = {p_val:.2e}')

# SMD balance check
print('\n=== Balance Check (SMD) ===')
t_matched = survived.iloc[[p[0] for p in matched_pairs]]
c_matched = survived.iloc[[p[1] for p in matched_pairs]]
for v in ['latency', 'SA_count', 'DA_count', 'total_touch_count', 'has_touchpoint', 'unique_channel_count']:
    if v in survived.columns:
        t_v = pd.to_numeric(t_matched[v], errors='coerce').fillna(0)
        c_v = pd.to_numeric(c_matched[v], errors='coerce').fillna(0)
        pooled = np.sqrt((t_v.std()**2 + c_v.std()**2) / 2)
        smd = abs(t_v.mean() - c_v.mean()) / pooled if pooled > 0 else 0
        print(f'  {v}: SMD={smd:.4f} {"✓" if smd < 0.1 else "✗"}')

In [ ]:
# Channel-specific feature importance
print('=== Channel-Specific Feature Importance ===')

for ch_name, ch_filter in [('DA', df_sample['last_touch_is_da'] == 1),
                           ('SA', df_sample['last_touch_is_sa'] == 1)]:
    sub = df_sample[ch_filter]
    if len(sub) > 1000:
        X_ch = sub[ua_cols].replace([np.inf, -np.inf], np.nan).fillna(0).values
        y_ch = sub['IS_D7_PURCHASE'].values
        
        rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
        rf.fit(X_ch, y_ch)
        imp = pd.Series(rf.feature_importances_, index=ua_cols).sort_values(ascending=False)
        
        print(f'\n{ch_name} users (N={len(sub):,}) - Top 5 features:')
        for name, val in imp.head(5).items():
            print(f'  {name}: {val:.4f}')

In [ ]:
# Channel × Latency interaction
print('=== Channel × Latency Interaction ===')

paid = df[df['has_touchpoint'] == 1].copy()
paid['latency_hours'] = paid['latency'] / 3600

lat_bins = [(0, 1, '<1h'), (1, 24, '1~24h'), (24, float('inf'), '24h+')]

print(f'{"Channel × Latency":<20} {"N":>8} {"D7 Purchase":>12}')
print('-' * 45)
for ch in ['DA', 'SA']:
    ch_data = paid[paid[f'last_touch_is_{ch.lower()}'] == 1]
    for low, high_b, label in lat_bins:
        sub = ch_data[(ch_data['latency_hours'] >= low) & (ch_data['latency_hours'] < high_b)]
        if len(sub) > 0:
            purch = sub['IS_D7_PURCHASE'].mean() * 100
            print(f'{ch} × {label:<12} {len(sub):>8,} {purch:>11.1f}%')

print(f'\nFraming: DA<1h=12.8%, DA 1-24h=9.3%, DA 24h+=14.0%')
print(f'Framing: SA<1h=15.9%, SA 1-24h=19.1%, SA 24h+=22.5%')

In [ ]:
# SA event count in first hour
print('=== SA vs DA: First Hour Core Events ===')

sa_users = df[df['channel'] == 'SA']
da_users = df[df['channel'] == 'DA']

if 'inapp_m60_core_engagement' in df.columns:
    sa_events = sa_users['inapp_m60_core_engagement'].mean()
    da_events = da_users['inapp_m60_core_engagement'].mean()
    print(f'SA first-hour core events: {sa_events:.2f}')
    print(f'DA first-hour core events: {da_events:.2f}')
    print(f'Difference: {(sa_events/da_events - 1)*100:+.0f}%')
    print(f'\nFraming expected: SA=7.84, DA=6.08, +29%')

---
## Phase 6: Visualization

Key plots for the research framing document.

In [ ]:
# Plot 1: 30-minute resolution churn and purchase curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(len(churn_rates)), churn_rates, 'o-', color='#e74c3c', markersize=4)
ax1.set_xticks(range(len(time_labels)))
ax1.set_xticklabels(time_labels, rotation=45, fontsize=8)
ax1.set_ylabel('Cumulative Churn Rate (%)')
ax1.set_title('Cumulative Churn Curve (30-min Resolution)')
ax1.grid(alpha=0.3)

ax2.plot(range(len(purchase_rates)), purchase_rates, 'o-', color='#27ae60', markersize=4)
ax2.set_xticks(range(len(time_labels)))
ax2.set_xticklabels(time_labels, rotation=45, fontsize=8)
ax2.set_ylabel('Cumulative Purchase Rate (%)')
ax2.set_title('Cumulative Purchase Curve (30-min Resolution)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_30min_churn_purchase_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_30min_churn_purchase_curves.png')

In [ ]:
# Plot 2: UA AUC decay at fine resolution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x = range(len(fine_decay_df))
ax1.plot(x, fine_decay_df['AUC_with_UA'], 'o-', color='#3498db', markersize=4, label='With UA')
ax1.plot(x, fine_decay_df['AUC_no_UA'], 's-', color='#95a5a6', markersize=4, label='Without UA')
ax1.fill_between(x, fine_decay_df['AUC_no_UA'], fine_decay_df['AUC_with_UA'], alpha=0.2, color='#3498db')
ax1.set_xticks(x)
ax1.set_xticklabels(fine_decay_df['window'], rotation=45, fontsize=8)
ax1.set_ylabel('AUC-ROC (D30 Purchase)')
ax1.set_title('AUC With vs Without UA (19 Time Points)')
ax1.legend()
ax1.grid(alpha=0.3)

bars = ax2.bar(x, fine_decay_df['delta_UA'], color='#e67e22', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(fine_decay_df['window'], rotation=45, fontsize=8)
ax2.set_ylabel('ΔUA (AUC Lift)')
ax2.set_title('Incremental UA Value Over Time')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_ua_auc_decay_fine.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_ua_auc_decay_fine.png')

In [ ]:
# Plot 3: Feature importance transition
fig, ax = plt.subplots(figsize=(12, 6))

x = range(len(fi_df))
ax.stackplot(x, fi_df['device'], fi_df['ua'], fi_df['inapp'],
             labels=['Device', 'UA (Ad Journey)', 'InApp'],
             colors=['#95a5a6', '#3498db', '#e74c3c'], alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(fi_df['window'], rotation=45)
ax.set_ylabel('Feature Importance Share (%)')
ax.set_title('Information Source Transition Over Time')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fig_feature_importance_transition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_feature_importance_transition.png')

---
## Summary: All Numbers Collected

All verified numbers from this notebook will be used to update `research_framing_kor.md`.

In [ ]:
print('=== ANALYSIS COMPLETE ===')
print(f'Dataset: coldstart_dataset_260304.csv')
print(f'Total users: {len(df_new):,}')
print(f'Analysis population (fraud excluded): {len(df):,}')
print(f'Columns: {df_new.shape[1]}')
print(f'\nAll Phase 2 numbers verified against research_framing_kor.md')
print(f'Phase 3: 30-minute resolution analysis complete')
print(f'Phase 4: LTV analysis complete')
print(f'Phase 5: Creative analysis verified')